In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/creditcardfraud/creditcard.csv


### Read/Load the data

In [2]:
df = pd.read_csv(r'/kaggle/input/creditcardfraud/creditcard.csv')

### For Simplicity , we are dropping Time Column

In [3]:
df = df.drop('Time' , axis = 1)

In [4]:
X = df.drop('Class' , axis =1)
y = df['Class']


### We Normalize the Data so that the each column values are mapped between 0 and 1
### This helps ML algorithms ( special the one which use distance metrics) converge faster

In [5]:
from sklearn import preprocessing

scaler = preprocessing.MinMaxScaler()


def scaleColumns(df, cols_to_scale, fit= True):
    for col in cols_to_scale:
        if fit == True :
            df[col] = pd.DataFrame(scaler.fit_transform(pd.DataFrame(df[col])),columns=[col])
        else :
            df[col] = pd.DataFrame(scaler.transform(pd.DataFrame(df[col])),columns=[col])
    return df

In [6]:
X = scaleColumns(X, list(X.columns))
X.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,0.935192,0.766490,0.881365,0.313023,0.763439,0.267669,0.266815,0.786444,0.475312,0.510600,...,0.582942,0.561184,0.522992,0.663793,0.391253,0.585122,0.394557,0.418976,0.312697,0.005824
1,0.978542,0.770067,0.840298,0.271796,0.766120,0.262192,0.264875,0.786298,0.453981,0.505267,...,0.579530,0.557840,0.480237,0.666938,0.336440,0.587290,0.446013,0.416345,0.313423,0.000105
2,0.935217,0.753118,0.868141,0.268766,0.762329,0.281122,0.270177,0.788042,0.410603,0.513018,...,0.585855,0.565477,0.546030,0.678939,0.289354,0.559515,0.402727,0.415489,0.311911,0.014739
3,0.941878,0.765304,0.868484,0.213661,0.765647,0.275559,0.266803,0.789434,0.414999,0.507585,...,0.578050,0.559734,0.510277,0.662607,0.223826,0.614245,0.389197,0.417669,0.314371,0.004807
4,0.938617,0.776520,0.864251,0.269796,0.762975,0.263984,0.268968,0.782484,0.490950,0.524303,...,0.584615,0.561327,0.547271,0.663392,0.401270,0.566343,0.507497,0.420561,0.317490,0.002724


### please note the difference - we are doing 50 - 50 split here

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
train_df = X.copy()
train_df['Class'] = y

### Initialize some random vector from the dataset

In [10]:
# Vector Initialization

no_of_vectors = 20
event_vectors = 3
non_event_vectors = no_of_vectors - event_vectors

print(event_vectors, non_event_vectors)

3 17


In [11]:
import numpy as np
import random 

df = df.sample(frac = 1)

event_vectors = train_df[train_df['Class'] ==1].head(event_vectors)
non_event_vectors = train_df[train_df['Class'] ==0].head(non_event_vectors)


### Lets see the vectors

In [12]:
final_vector = pd.concat([event_vectors, non_event_vectors], axis =0 , ignore_index = True)

final_vector

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.919012,0.787855,0.809517,0.429154,0.762201,0.248677,0.249897,0.800314,0.367355,0.451365,...,0.569817,0.508396,0.658525,0.425381,0.580406,0.454498,0.421331,0.310216,0.000000,1
1,0.906588,0.733944,0.856275,0.353384,0.774870,0.252314,0.267339,0.784658,0.453446,0.491372,...,0.572146,0.530346,0.685868,0.342644,0.593612,0.401704,0.411845,0.313850,0.020591,1
2,0.919163,0.785821,0.831180,0.355228,0.760185,0.262258,0.268781,0.781104,0.454573,0.477162,...,0.556737,0.466535,0.667999,0.370467,0.569143,0.336811,0.417241,0.310018,0.009339,1
3,0.935192,0.766490,0.881365,0.313023,0.763439,0.267669,0.266815,0.786444,0.475312,0.510600,...,0.561184,0.522992,0.663793,0.391253,0.585122,0.394557,0.418976,0.312697,0.005824,0
4,0.978542,0.770067,0.840298,0.271796,0.766120,0.262192,0.264875,0.786298,0.453981,0.505267,...,0.557840,0.480237,0.666938,0.336440,0.587290,0.446013,0.416345,0.313423,0.000105,0
5,0.935217,0.753118,0.868141,0.268766,0.762329,0.281122,0.270177,0.788042,0.410603,0.513018,...,0.565477,0.546030,0.678939,0.289354,0.559515,0.402727,0.415489,0.311911,0.014739,0
6,0.941878,0.765304,0.868484,0.213661,0.765647,0.275559,0.266803,0.789434,0.414999,0.507585,...,0.559734,0.510277,0.662607,0.223826,0.614245,0.389197,0.417669,0.314371,0.004807,0
7,0.938617,0.776520,0.864251,0.269796,0.762975,0.263984,0.268968,0.782484,0.490950,0.524303,...,0.561327,0.547271,0.663392,0.401270,0.566343,0.507497,0.420561,0.317490,0.002724,0
8,0.951057,0.777393,0.857187,0.244472,0.768550,0.262721,0.268257,0.788178,0.443190,0.501038,...,0.558122,0.483915,0.665042,0.332185,0.564839,0.442749,0.421196,0.314769,0.000143,0
9,0.979184,0.768746,0.838200,0.305241,0.767008,0.265762,0.265324,0.786257,0.478797,0.506668,...,0.558776,0.497402,0.663145,0.277122,0.620014,0.383429,0.417148,0.313229,0.000194,0


In [13]:
final_vector_orig = final_vector.copy()

# This is where the model training happens 

## num_epoch --> how many times we want to loop over the training data

## pdist and sqaureform are used to calculate the distance between our training records and the 20 vectors 

## update_vector -->  it will a number between 0 and 19 - indicating the vector which we need to update

## alpha --> it control by how much we must update the vectors 

## 

In [14]:
# Training Started 
import time
start_time = time.time()
alpha = 0.01
num_epoch = 3
from scipy.spatial.distance import pdist, squareform
final_vector = final_vector_orig.drop('Class' , axis =1 )
for j in range(num_epoch) :
    for i in range(len(X_train)) :
        only_vector = final_vector.copy()
        a = X_train.iloc[i,:].tolist()
        #print(type(a))
        only_vector.loc[only_vector.shape[0]] =a 
        #print(only_vector.tail())
        distances = pdist(only_vector, metric='euclidean')
        dist_matrix = squareform(distances)
        b = np.array(dist_matrix[only_vector.shape[0] -1])[:-1]
        update_vector = b.argmin()
        #print(update_vector)
        vector_type = final_vector_orig.iloc[update_vector, -1]
        actual_type = y_train.iloc[i]
        #print(vector_type ,actual_type )

        vector_to_update = only_vector.iloc[update_vector].values
        #print(vector_to_update)
        diff = alpha *(X_train.iloc[i] - vector_to_update)
        #print(diff)

        if vector_type ==  actual_type :
            vector_to_update = vector_to_update  + diff
        else :
            vector_to_update = vector_to_update  - diff


        final_vector.loc[update_vector] =list(vector_to_update)


        #print(final_vector.isna().sum())
    print("Epoch Completed : " , j+1 )
    
    
end_time =  time.time()  

print("Total Time in Seconds : ", str(round(end_time - start_time , 3)))

Epoch Completed :  1
Epoch Completed :  2
Epoch Completed :  3
Total Time in Seconds :  1062.463


In [15]:
final_vector

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,0.843473,0.827921,0.670055,0.541067,0.739627,0.236305,0.202298,0.802688,0.312695,0.339046,...,0.592588,0.568858,0.526318,0.664632,0.364934,0.569123,0.460130,0.408379,0.316059,-0.008058
1,0.961748,0.762894,0.940228,0.326093,0.791254,0.234464,0.258725,0.786938,0.462879,0.462168,...,0.622744,0.585529,0.570128,0.697048,0.358945,0.613661,0.364767,0.399812,0.320846,0.003120
2,0.916419,0.793542,0.775705,0.398257,0.755611,0.255154,0.250034,0.781507,0.416212,0.431431,...,0.581980,0.566624,0.498450,0.666403,0.376596,0.575754,0.403750,0.418266,0.312712,0.006984
3,0.963042,0.768643,0.843973,0.268880,0.763296,0.258901,0.265886,0.784902,0.473709,0.506384,...,0.579950,0.561092,0.510996,0.665011,0.395490,0.582680,0.374898,0.416476,0.313035,0.002898
4,0.949903,0.762736,0.831348,0.216622,0.769714,0.276033,0.265410,0.788607,0.469114,0.508512,...,0.577956,0.562383,0.526408,0.665897,0.215009,0.562098,0.391341,0.414113,0.311137,0.004486
5,0.957151,0.764903,0.839355,0.249595,0.768890,0.275484,0.263309,0.790010,0.465363,0.505213,...,0.578195,0.563872,0.529736,0.667150,0.263207,0.562499,0.477203,0.417845,0.312689,0.002888
6,0.958891,0.768653,0.829680,0.265132,0.766152,0.259672,0.266217,0.784371,0.459826,0.502735,...,0.579634,0.561701,0.516725,0.665148,0.361829,0.581138,0.369459,0.417240,0.312493,0.004111
7,0.959409,0.765507,0.837483,0.225955,0.763645,0.262616,0.264319,0.784306,0.462752,0.509914,...,0.580511,0.560655,0.506504,0.665403,0.424357,0.575778,0.561098,0.415355,0.312469,0.004215
8,0.955850,0.769195,0.837166,0.236105,0.764687,0.258286,0.267102,0.785078,0.476351,0.505371,...,0.580985,0.560946,0.502998,0.665292,0.407089,0.575411,0.506380,0.416730,0.313530,0.003683
9,0.962281,0.767330,0.835120,0.236080,0.767229,0.265521,0.265347,0.786380,0.458265,0.510896,...,0.580681,0.561099,0.502858,0.664147,0.247667,0.582581,0.433553,0.416519,0.313086,0.002724


### save the vectors in a csv file

In [16]:
final_vector2 = final_vector.copy()
final_vector2['Tag'] = final_vector_orig['Class']
final_vector2.to_csv('vectors.csv')